In [18]:
import os
import pickle
import pandas as pd

METADATA_DIR = '/secure/shared_data/tcga_path_reports/TCGA_Metadata'
REPORTS_DIR  = '/secure/shared_data/tcga_path_reports/TCGA_Pathology_Reports'
OUT_DIR      = 'per_cancer_type'

os.makedirs(OUT_DIR, exist_ok=True)

cancer_type_df = pickle.load(open(os.path.join(METADATA_DIR, 'TCGA_cancer_types_binary.p'), 'rb'))
t14_raw = pd.read_csv(os.path.join(METADATA_DIR, 'TCGA_T14_patients.csv'))
n03_raw = pd.read_csv(os.path.join(METADATA_DIR, 'TCGA_N03_patients.csv'))

path_report_text = pd.read_csv(os.path.join(REPORTS_DIR, 'TCGA_Reports.csv'))

patients = (
    cancer_type_df[['patient', 'type', 'patient_filename']]
    .rename(columns={'patient': 'case_submitter_id'})
    .copy()
)

def map_t14(s):
    if pd.isna(s): return None
    s = str(s).upper()
    if s in ('T0', 'TX'):
        return None
    if '1' in s:
        return 0
    elif '2' in s:
        return 1
    elif '3' in s:
        return 2
    elif '4' in s:
        return 3
    return None  # unrecognized, report below

def map_n03(s):
    if pd.isna(s): return None
    s = str(s).upper()
    if s in ('NX', 'N0 (I+)', 'N0 (I-)', 'N0 (MOL+)'):
        return None
    if s == 'N0':
        return 0
    elif '1' in s:
        return 1
    elif '2' in s:
        return 2
    elif '3' in s:
        return 3
    return None

# ---------- Clean + encode each stage ----------
t14_stage = t14_raw[['case_submitter_id', 'ajcc_pathologic_t']].copy()
t14_stage['T14'] = t14_stage['ajcc_pathologic_t'].apply(map_t14)
# Report any unrecognized values (not T0/TX) before dropping:
unknown_t = sorted(
    set(t14_stage.loc[t14_stage['T14'].isna(), 'ajcc_pathologic_t']) - {'T0', 'TX'}
)
if unknown_t:
    print('[WARN] Unrecognized ajcc_pathologic_t values:', unknown_t)
t14_stage = t14_stage.dropna(subset=['T14']).drop_duplicates(subset=['case_submitter_id'])

n03_stage = n03_raw[['case_submitter_id', 'ajcc_pathologic_n']].copy()
n03_stage['N03'] = n03_stage['ajcc_pathologic_n'].apply(map_n03)
unknown_n = sorted(
    set(n03_stage.loc[n03_stage['N03'].isna(), 'ajcc_pathologic_n']) - {'NX', 'N0 (i+)', 'N0 (i-)', 'N0 (mol+)','N0 (I+)', 'N0 (I-)', 'N0 (MOL+)'}
)
if unknown_n:
    print('[WARN] Unrecognized ajcc_pathologic_n values:', unknown_n)
n03_stage = n03_stage.dropna(subset=['N03']).drop_duplicates(subset=['case_submitter_id'])


# ---------- Build the union set of patients that have at least one stage ----------
stage_ids = set(t14_stage['case_submitter_id']) | set(n03_stage['case_submitter_id'])
patients_stage = patients[patients['case_submitter_id'].isin(stage_ids)].copy()

# ---------- Merge stage annotations onto the patient/type mapping ----------
annot = (
    patients_stage
    .merge(t14_stage, on='case_submitter_id', how='left')
    .merge(n03_stage, on='case_submitter_id', how='left')
)

annot = annot[['type',
               'case_submitter_id', 'patient_filename',
               'ajcc_pathologic_t', 'T14',
               'ajcc_pathologic_n', 'N03']]

annot = annot.merge(path_report_text, on='patient_filename', how='left')  

per_type_counts = []
for ctype, df_ct in annot.groupby('type', sort=True):
    out_path = os.path.join(OUT_DIR, f'{ctype}_T14N03.csv')
    df_ct.sort_values(['case_submitter_id'], inplace=True)
    df_ct.to_csv(out_path, index=False)
    per_type_counts.append({
        'type': ctype,
        'rows': len(df_ct),
        'with_T14': int(df_ct['T14'].notna().sum()),
        'with_N03': int(df_ct['N03'].notna().sum()),
    })

annot.sort_values(['type', 'case_submitter_id']).to_csv(os.path.join(OUT_DIR, 'ALL_T14N03_by_cancer_type.csv'), index=False)
summary_df = pd.DataFrame(per_type_counts).sort_values('type')
summary_df.to_csv(os.path.join(OUT_DIR, 'SUMMARY_counts_per_type.csv'), index=False)

print(f'Wrote {summary_df.shape[0]} per-cancer CSVs to: {OUT_DIR}')
print('Master file:', os.path.join(OUT_DIR, 'ALL_T14N03_by_cancer_type.csv'))
print('Summary file:', os.path.join(OUT_DIR, 'SUMMARY_counts_per_type.csv'))


Wrote 23 per-cancer CSVs to: per_cancer_type
Master file: per_cancer_type/ALL_T14N03_by_cancer_type.csv
Summary file: per_cancer_type/SUMMARY_counts_per_type.csv


In [ ]:
# # M included
# import os
# import pickle
# import pandas as pd

# METADATA_DIR = '/secure/shared_data/tcga_path_reports/TCGA_Metadata'
# REPORTS_DIR  = '/secure/shared_data/tcga_path_reports/TCGA_Pathology_Reports'
# OUT_DIR      = 'per_cancer_type'

# os.makedirs(OUT_DIR, exist_ok=True)


# cancer_type_df = pickle.load(open(os.path.join(METADATA_DIR, 'TCGA_cancer_types_binary.p'), 'rb'))
# t14_raw = pd.read_csv(os.path.join(METADATA_DIR, 'TCGA_T14_patients.csv'))
# n03_raw = pd.read_csv(os.path.join(METADATA_DIR, 'TCGA_N03_patients.csv'))
# m01_raw = pd.read_csv(os.path.join(METADATA_DIR, 'TCGA_M01_patients.csv'))

# path_report_text = pd.read_csv(os.path.join(REPORTS_DIR, 'TCGA_Reports.csv'))

# patients = (
#     cancer_type_df[['patient', 'type', 'patient_filename']]
#     .rename(columns={'patient': 'case_submitter_id'})
#     .copy()
# )


# def map_t14(s):
#     if pd.isna(s): return None
#     s = str(s).upper()
#     if s in ('T0', 'TX'):
#         return None
#     if '1' in s:
#         return 0
#     elif '2' in s:
#         return 1
#     elif '3' in s:
#         return 2
#     elif '4' in s:
#         return 3
#     return None  # unrecognized, report below

# def map_n03(s):
#     if pd.isna(s): return None
#     s = str(s).upper()
#     if s in ('NX', 'N0 (I+)', 'N0 (I-)', 'N0 (MOL+)'):
#         return None
#     if s == 'N0':
#         return 0
#     elif '1' in s:
#         return 1
#     elif '2' in s:
#         return 2
#     elif '3' in s:
#         return 3
#     return None

# def map_m01(s):
#     if pd.isna(s): return None
#     s = str(s).upper()
#     if s == 'MX':
#         return None
#     if '0' in s:
#         return 0
#     elif '1' in s:
#         return 1
#     return None


# t14_stage = t14_raw[['case_submitter_id', 'ajcc_pathologic_t']].copy()
# t14_stage['T14'] = t14_stage['ajcc_pathologic_t'].apply(map_t14)
# # Report any unrecognized values (not T0/TX) before dropping:
# unknown_t = sorted(
#     set(t14_stage.loc[t14_stage['T14'].isna(), 'ajcc_pathologic_t']) - {'T0', 'TX'}
# )
# if unknown_t:
#     print('[WARN] Unrecognized ajcc_pathologic_t values:', unknown_t)
# t14_stage = t14_stage.dropna(subset=['T14']).drop_duplicates(subset=['case_submitter_id'])

# n03_stage = n03_raw[['case_submitter_id', 'ajcc_pathologic_n']].copy()
# n03_stage['N03'] = n03_stage['ajcc_pathologic_n'].apply(map_n03)
# unknown_n = sorted(
#     set(n03_stage.loc[n03_stage['N03'].isna(), 'ajcc_pathologic_n']) - {'NX', 'N0 (i+)', 'N0 (i-)', 'N0 (mol+)','N0 (I+)', 'N0 (I-)', 'N0 (MOL+)'}
# )
# if unknown_n:
#     print('[WARN] Unrecognized ajcc_pathologic_n values:', unknown_n)
# n03_stage = n03_stage.dropna(subset=['N03']).drop_duplicates(subset=['case_submitter_id'])

# m01_stage = m01_raw[['case_submitter_id', 'ajcc_pathologic_m']].copy()
# m01_stage['M01'] = m01_stage['ajcc_pathologic_m'].apply(map_m01)
# unknown_m = sorted(
#     set(m01_stage.loc[m01_stage['M01'].isna(), 'ajcc_pathologic_m']) - {'MX'}
# )
# if unknown_m:
#     print('[WARN] Unrecognized ajcc_pathologic_m values:', unknown_m)
# m01_stage = m01_stage.dropna(subset=['M01']).drop_duplicates(subset=['case_submitter_id'])

# stage_ids = set(t14_stage['case_submitter_id']) | set(n03_stage['case_submitter_id']) | set(m01_stage['case_submitter_id'])
# patients_stage = patients[patients['case_submitter_id'].isin(stage_ids)].copy()


# annot = (
#     patients_stage
#     .merge(t14_stage, on='case_submitter_id', how='left')
#     .merge(n03_stage, on='case_submitter_id', how='left')
#     .merge(m01_stage, on='case_submitter_id', how='left')
# )

# # Optional: if you want only rows with all three labels present, uncomment:
# # annot = annot.dropna(subset=['T14', 'N03', 'M01'])

# # Select and order the final columns
# annot = annot[['type',
#                'case_submitter_id', 'patient_filename',
#                'ajcc_pathologic_t', 'T14',
#                'ajcc_pathologic_n', 'N03',
#                'ajcc_pathologic_m', 'M01']]


# annot = annot.merge(path_report_text, on='patient_filename', how='left')  # adds 'text' column



# per_type_counts = []
# for ctype, df_ct in annot.groupby('type', sort=True):
#     out_path = os.path.join(OUT_DIR, f'{ctype}_T14N03M01.csv')
#     df_ct.sort_values(['case_submitter_id'], inplace=True)
#     df_ct.to_csv(out_path, index=False)
#     per_type_counts.append({
#         'type': ctype,
#         'rows': len(df_ct),
#         'with_T14': int(df_ct['T14'].notna().sum()),
#         'with_N03': int(df_ct['N03'].notna().sum()),
#         'with_M01': int(df_ct['M01'].notna().sum()),
#     })


# annot.sort_values(['type', 'case_submitter_id']).to_csv(os.path.join(OUT_DIR, 'ALL_T14N03M01_by_cancer_type.csv'), index=False)
# summary_df = pd.DataFrame(per_type_counts).sort_values('type')
# summary_df.to_csv(os.path.join(OUT_DIR, 'SUMMARY_counts_per_type.csv'), index=False)

# print(f'Wrote {summary_df.shape[0]} per-cancer CSVs to: {OUT_DIR}')
# print('Master file:', os.path.join(OUT_DIR, 'ALL_T14N03M01_by_cancer_type.csv'))
# print('Summary file:', os.path.join(OUT_DIR, 'SUMMARY_counts_per_type.csv'))


Wrote 23 per-cancer CSVs to: per_cancer_type
Master file: per_cancer_type/ALL_T14N03M01_by_cancer_type.csv
Summary file: per_cancer_type/SUMMARY_counts_per_type.csv


In [20]:
df = pd.read_csv("/home/yl3427/cylab/selfCorrectionAgent/per_cancer_type/ALL_T14N03_by_cancer_type.csv")
print(df.type.nunique())
len(df[df['T14'].notna()].type.unique()), len(df[df['N03'].notna()].type.unique())

23


(23, 23)

In [17]:
set(df[df['T14'].notna()].type.unique()) == (set(df[df['N03'].notna()].type.unique()))

True

https://gdc.cancer.gov/resources-tcga-users/tcga-code-tables/tcga-study-abbreviations

In [24]:
# ['BLCA', 'UCS', 'HNSC', 'STAD',
#        'CESC', 'DLBC', 'KIRC', 'UCEC', 'PRAD', 'SARC', 'THYM', 'KIRP', 'KICH',
#        'LIHC', 'BRCA', 'LUAD', 'PAAD', 'OV', 'THCA', 'MESO', 'ACC', 'CHOL',
#        'TGCT', 'LUSC', 'READ', 'SKCM', 'COAD', 'PCPG', 'UVM', 'ESCA', 'GBM',
#        'LGG']
cancer_type_map = {
    'BLCA': 'Bladder Urothelial Carcinoma',
    'UCS': 'Uterine Carcinosarcoma',
    'HNSC': 'Head and Neck Squamous Cell Carcinoma',
    'STAD': 'Stomach Adenocarcinoma',
    'CESC': 'Cervical Squamous Cell Carcinoma and Endocervical Adenocarcinoma',
    'DLBC': 'Diffuse Large B-cell Lymphoma',
    'KIRC': 'Kidney Renal Clear Cell Carcinoma',
    'UCEC': 'Uterine Corpus Endometrial Carcinoma',
    'PRAD': 'Prostate Adenocarcinoma',
    'SARC': 'Sarcoma',
    'THYM': 'Thymoma',
    'KIRP': 'Kidney Renal Papillary Cell Carcinoma',
    'KICH': 'Kidney Chromophobe',
    'LIHC': 'Liver Hepatocellular Carcinoma',
    'BRCA': 'Breast Invasive Carcinoma',
    'LUAD': 'Lung Adenocarcinoma',
    'PAAD': 'Pancreatic Adenocarcinoma',
    'OV': 'Ovarian Serous Cystadenocarcinoma',
    'THCA': 'Thyroid Carcinoma',
    'MESO': 'Mesothelioma',
    'ACC': 'Adrenocortical Carcinoma',
    'CHOL': 'Cholangiocarcinoma',
    'TGCT': 'Testicular Germ Cell Tumors',
    'LUSC': 'Lung Squamous Cell Carcinoma',
    'READ': 'Rectum Adenocarcinoma',
    'SKCM': 'Skin Cutaneous Melanoma',
    'COAD': 'Colon Adenocarcinoma',
    'PCPG': 'Pheochromocytoma and Paraganglioma',
    'UVM': 'Uveal Melanoma',
    'ESCA': 'Esophageal Carcinoma',
    'GBM': 'Glioblastoma Multiforme',
    'LGG': 'Brain Lower Grade Glioma'
}
len(cancer_type_map)

32

In [25]:
df.type.unique()
cancer_type_map = {k: v for k, v in cancer_type_map.items() if k in df.type.unique()}
cancer_type_map

{'BLCA': 'Bladder Urothelial Carcinoma',
 'HNSC': 'Head and Neck Squamous Cell Carcinoma',
 'STAD': 'Stomach Adenocarcinoma',
 'CESC': 'Cervical Squamous Cell Carcinoma and Endocervical Adenocarcinoma',
 'KIRC': 'Kidney Renal Clear Cell Carcinoma',
 'PRAD': 'Prostate Adenocarcinoma',
 'KIRP': 'Kidney Renal Papillary Cell Carcinoma',
 'KICH': 'Kidney Chromophobe',
 'LIHC': 'Liver Hepatocellular Carcinoma',
 'BRCA': 'Breast Invasive Carcinoma',
 'LUAD': 'Lung Adenocarcinoma',
 'PAAD': 'Pancreatic Adenocarcinoma',
 'THCA': 'Thyroid Carcinoma',
 'MESO': 'Mesothelioma',
 'ACC': 'Adrenocortical Carcinoma',
 'CHOL': 'Cholangiocarcinoma',
 'TGCT': 'Testicular Germ Cell Tumors',
 'LUSC': 'Lung Squamous Cell Carcinoma',
 'READ': 'Rectum Adenocarcinoma',
 'SKCM': 'Skin Cutaneous Melanoma',
 'COAD': 'Colon Adenocarcinoma',
 'UVM': 'Uveal Melanoma',
 'ESCA': 'Esophageal Carcinoma'}

In [27]:
len(cancer_type_map)

23